<a class="anchor" id="Top"></a>
# <span style="color:DodgerBlue">Calculating dN/dS Ratios For Single Copy Orthologs In Culex [1:1:1:1]</span>
***
**In this file:** [Creating FNAs](#0B) | [Finding Corresponding FNA Sequences](#1B) | [dN/dS Ratios](#2B) | [Methods/Example](#3B) | [Analysis](#4B)
***
<a class="anchor" id="0B"></a>
### <span style="color:DodgerBlue">Creating FNAs From GFF3s + FAAs</span>
Genome fna files were provided for three of the four Culex genomes. The only genome without a fna file was the Davis Culex tarsalis genome.
#### gffread_script.slurm
Cufflinks version 2.2.1 (gffread) was used to create a fna file for the Davis Culex tarsalis genome using its contigs fasta and gff3.

In [17]:
%%script bash --bg
module load cufflinks/

gffread -w Davis_tarsalis_AnnotatedTranscripts.fasta -g Culex-tarsalis_knwr_CONTIGS_CtarK1.fa \
Culex-tarsalis_knwr_BASEFEATURES_CtarK1.gff3

#### fnafileadjustments_script.py
A custom python3 script was created to adjust the original fna ortholog files (adding the species to each gene name + making gene names uniform).

In [ ]:
%%script python3 --bg
with open('Clemson_Culex-tarsalis-v1.0.a1.5d67b82c92619-publish.CDS.fna') as file:
    with open('Clemson_tarsalis_transcript.fa','w') as outfile:
        for line in file:
            line = line.strip()
            if line.startswith('>Ct.'):
                nline = line.replace('>Ct.',">Clemson_tarsalis_Ct.")
                outfile.write(nline+'\n')
            else:
                outfile.write(line+'\n')

with open('VectorBase-58_CquinquefasciatusJohannesburg_AnnotatedTranscripts.fasta') as file:
    with open('Culex_quinquefasciatus_transcript.fa','w') as outfile:
        for line in file:
            line = line.strip()
            if line.startswith('>CPI'):
                nline = line.replace('>CPI',">Culex_quinquefasciatus_CPI")
                outfile.write(nline+'\n')
            else:
                outfile.write(line+'\n')

with open('Davis_tarsalis_AnnotatedTranscripts.fasta') as file:
    with open('Davis_tarsalis_transcript.fa','w') as outfile:
        for line in file:
            line = line.strip()
            if line.startswith('>mRNA'):
                nline = line.replace('>mRNA',">Davis_tarsalis_mRNA")
                outfile.write(nline+'\n')
            else:
                if not line.startswith('>B48'):
                    outfile.write(line+'\n')

#### cat_script.slurm
A custom bash script was written to concatinate the transcript files for all 4 genomes.

In [ ]:
%%script bash --bg
mv proteins.Culex_pipiens_pallens.fasta Culex_pipiens_transcript.fa

cat Clemson_tarsalis_transcript.fa Culex_quinquefasciatus_transcript.fa \
Culex_pipiens_transcript.fa Davis_tarsalis_transcript.fa \
>> Culex_transcripts.fa

[Back To Top](#Top)
***
<a class="anchor" id="1B"></a>
### <span style="color:DodgerBlue">Finding Corresponding FNA Sequences For The Single Copy Ortholog FAA Sequences</span>
Custom bash scripts were utilized to make new single copy ortholog fastas in fna format (they are presently in faa format).
Script findfasta.awk was taken from https://stackoverflow.com/questions/34971403/retrieving-dna-sequences-from-a-fasta-file-using-ids.
#### findfasta.awk

In [12]:
%%script bash --bg
BEGIN {
    while ( getline < "-" ) {
        selected[$0] = 1
    }
    RS=">"
}
$1 in selected {
    printf ">%s", $0
}

#### ForAllFilenames.sh
A bash script was created to copy the names of all single copy ortholog fastas into a file called 'All_filenames.txt', to later have its lines read in as a variable.

In [2]:
%%script bash --bg
for infile in OG0*
do
     echo ${infile} >> All_filenames.txt
done

#### ForEachGeneList.py
A python3 script that makes a new file of all gene names in each single copy ortholog fasta, along with unifying the gene names for ease of use later.

In [15]:
%%script python3 --bg
with open('All_filenames.txt') as allfile:
    for line in allfile:
        with open(line[:-1]) as infile:
            with open(line[:-4] + '_gene_list.fa','w') as outfile:
                for l in infile:
                    if l.startswith('>'):
                        l = l.replace('>','')
                        if l.startswith('Clemson_tarsalis_'):
                            l = l.replace('.polypeptide','.CDS')
                            outfile.write(l[:-1]+'\n') 
                        elif l.startswith('Culex_quinquefasciatus_'):
                            l = l.replace('-PA','-RA')
                            outfile.write(l[:-1]+'\n') 
                        else:
                            outfile.write(l[:-1]+'\n')

#### ForAllTranscripts.sh
Bash script that utalizes findfasta.awk to create a new file similar to each of the single copy ortholog fastas that came out of OrthoFinder, but in fna format instead of faa. Names changes occur, and the files are zipped.

In [16]:
%%script bash --bg
for infile in *_gene_list.fa
do
    base=$(basename ${infile} _gene_list.fa)
    dos2unix ${infile}
    cat ${infile} | awk -f findfasta.awk Culex_transcripts.fa >> ${base}.fna
done

zip SingleCopyOrthologGeneListFiles.zip *_gene_list.fa
zip SingleCopyOrthologFnaFiles.zip *.fna

for infile in *.fa
do
    base=$(basename ${infile} .fa)
    mv ${infile} ${base}.faa
done

zip SingleCopyOrthologFaaFiles.zip *.faa

[Back To Top](#Top)
***
<a class="anchor" id="2B"></a>
### <span style="color:DodgerBlue">Detecting Pairwise dN/dS Ratios With Codeml (PAML)</span>
Codeml, a component of the PAML package, takes in an alignment of homologous coding regions and reads a control file that contains the run parameters. The protocol followed can be located at: https://www.protocols.io/view/introduction-to-calculating-dn-ds-ratios-with-code-3byl4o4rgo5d/v2?step=3, and a summary of the analtsis process can be found at: https://dwheelerau.com/2012/11/21/using-paml-to-detect-pairwise-dnds-ratios/.
***
The programs used in this tutorial are:

**codeml in the PAML package.** On a Ubuntu 16.04 LTS system it should be able to install this tool with "sudo apt install paml". http://abacus.gene.ucl.ac.uk/software/paml.html
<br>**PAL2NAL.** This is essentially a PERL script that you will want to have handy, either by putting it in the folder that you are working in or by putting it somewhere that is in your PATH. http://www.bork.embl.de/pal2nal/
<br>**clustal-omaga.** You should be able to install this with "sudo apt install clustalo" This is a nice amino acid and nucleic acid alignement program. For purposes here your choice of aligner is not critical, so if you prefer MAFFT or Muscle or something else you can continue using those. http://www.clustal.org/omega/ 
#### dNdS_script.sh

In [ ]:
%%script bash --bg

########## align the amino acid sequences using clustal omega ##########

for infile in *.faa
do
    base=$(basename ${infile} .faa)
    clustalo -i ${infile} -o ${base}.aln.faa
done

########## use pal2nal to get a codon-based nucleic acid alignment ##########

for infile in *.aln.faa
do
    base=$(basename ${infile} .aln.faa)
    pal2nal.pl ${infile} ${base}.fna -output paml -nogap > ${base}.pal2nal
done

cp codeml.ctl 1codeml.ctl

########## loop for codeml ##########

for infile in *.pal2nal
do
    base=$(basename ${infile} .pal2nal)
    mkdir ${base}_outputs
    cp -r parse_codeml_output.py ${infile} for_paml 1codeml.ctl ${base}-PhyML.tree test.* ${base}_outputs
    cd ${base}_outputs
    sed 's/cluster_1/'${base}'/' 1codeml.ctl > codeml.ctl
    rm 1codeml.ctl
    codeml
    python parse_codeml_output.py codeml.txt > ${base}.output
    cp ${base}.output ..
    cd ..
done

rm 1codeml.ctl

#### Final Results Example

In [30]:
OG0007727_output = pd.read_csv("OG0007727.output", sep='\t')
OG0007727_output

,Gene_1,Gene_2,dnds,dN,dS
0,Culex_pipiens_00014632-RA,Clemson_tarsalis_Ct.00g145080.m01.polypeptide,0.0697,0.0494,0.7084
1,Culex_quinquefasciatus_CPIJ009106-PA,Clemson_tarsalis_Ct.00g145080.m01.polypeptide,0.0704,0.0494,0.7017
2,Culex_quinquefasciatus_CPIJ009106-PA,Culex_pipiens_00014632-RA,0.0488,0.0014,0.0287
3,Davis_tarsalis_mRNA2982,Clemson_tarsalis_Ct.00g145080.m01.polypeptide,0.1052,0.0148,0.1405
4,Davis_tarsalis_mRNA2982,Culex_pipiens_00014632-RA,0.0626,0.0447,0.7136
5,Davis_tarsalis_mRNA2982,Culex_quinquefasciatus_CPIJ009106-PA,0.0633,0.0447,0.7065


>*Note that not all gene-pairs will be printed out. This is because the script filters out all pairs for which dS was < 0.01 or > 2. Values < 0.01 indicate that we may not get a reliable estimate of dN/dS, since the sequences are so similar. dS values > 2 indicate that the sequences are quite divergent and multiple substitutions have likely occured at most sites, so dN/dS estimates will again be compromised.*

[Back To Top](#Top)
***
<a class="anchor" id="3B"></a>
### <span style="color:DodgerBlue">Methods & Process Example</span>
For this section, faa, fna, aligned faa, and pal2nal files were used/produced. There were around 4,000 of them, and they look as follows:
#### Original, single copy ortholog faa that came out of OrthoFinder

In [22]:
N = 45
with open("OG0009798.faa") as OG0009798_faa:
    for i in range(N):
        line = next(OG0009798_faa).strip()
        if line.startswith('>'):
            print('\n' + line)
        else:
            print(line)


>Clemson_tarsalis_Ct.00g367850.m01.polypeptide
LRDPGAVGVAGITGSGPPPGMGTSGASSGRDQGKGLLAMEPQEFCDKPLI
LKIAIQTSGRTRCWCRGRAGSIPGCTVHVIFHSLDTVEDIRGKTLNRQAN
SAQLDRLSANTRYLICVLGLGNWLSGYHDHDIHSLLNQSNQIQNQVLNGP
HGYGVGQDGGNELDTSLSNSLLSLMMDTPTSRCTEVRTLDAIGPNPLAEV
DGMSSRSIIHSILTRRLGLIVGCCLGIIVFIVLISVLGYLKIKKQRLDAA
KRLQQPPMAPEFISYRHFSIPNDEHGRDGVAGVVGGGGTNTFLQGATVVA
NHDGHPSFISGAVLGTTTTLNGGAGTGLVPGEERKKIMFDS

>Culex_pipiens_00001482-RA
MMLRWWPNLGLTQPYYKYLWRCWLLVGLILLQRISYGQMEEILRLRGGGQGSGGGGGGSGHHGHHQQQHLQQLDEQEHHFSSASSNEIECPSFVDNSACPCYKFEDGLFLECPSITAVVLRSTLQLISSPIQSLSVYEFDRSVKSLTVDLFAPANQQSSDVNIRHLQFSNSNLQQLKENSLSNLRAHLESLSIVNGKLTQVPTKALAGLKKLMVLDFELNEISAIEEYAFYGLHLVKLNMKGNRLERIPENAFSGLEDSLAELDLSENRLKQFPTGALKRLENLRSVRLSMNEINSLEQDDSYTRFGSLVFLDLSLNNFVELYSDVFNPFPYVKTLSLYNNFIELVHRDSFVSLKELQSLDLSHNQVVFVDPEVFSANRKLHTVDLSHNHIHYVSGVFANLPLLREIFLSENNILELTDDCFSNSSSIKVIYLENNSLQRLGSDTLATVTNLEQLYLSGNHIQRIPVGFFETTVKLQSLSLDGNELTELDVRLFRRLANLREVRLNGNQLRSIREHLFAAQENMMELHLQNNVISLIERNAFKNCLQLQYINLQENELDEIDILLSTTASTDSANQ

#### Aligned, single copy ortholog faa that was output from the clustalo step

In [21]:
N = 45
with open("OG0009798.aln.faa") as OG0009798_aln_faa:
    for i in range(N):
        line = next(OG0009798_aln_faa).strip()
        if line.startswith('>'):
            print('\n' + line)
        else:
            print(line)


>Clemson_tarsalis_Ct.00g367850.m01.polypeptide
------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
-------------------------------------

#### Codon-based nucleic acid alignment, after inputting the aligned amino acid sequences and the raw nucleic acid sequences

In [24]:
N = 45
with open("OG0009798.pal2nal") as OG0009798_pal2nal:
    for i in range(N):
        line = next(OG0009798_pal2nal).strip()
        if line.startswith('Culex') or line.startswith('Davis') or line.startswith('Clemson'):
            print('\n' + line)
        else:
            print(line)

4   1002

Clemson_tarsalis_Ct.00g367850.m01.polypeptide
CTGCGAGACCCAGGAGCTGTGGGAGTGGCTGGGATCACCGGAAGTGGACCACCACCGGGT
ATGGGAACGTCCGGTGCGAGCAGCGGTCGAGATCAGGGCAAGGGTCTGCTGGCGATGGAG
CCGCAGGAGTTTTGCGACAAACCGTTGATACTGAAGATTGCCATCCAGACATCCGGCCGT
ACTCGGTGCTGGTGTCGTGGCAGAGCCGGGAGCATTCCGACGGTACACGTGATTTTCCAC
TCGCTCGACACGGTGGAGGATATCCGTGGCAAAACGCTCAACCGGCAGGCCAACTCGGCC
CAGCTAGACCGGCTGTCGGCCAACACGCGCTACCTAATCTGCGTCCTCGGTCTGGGCAAC
TGGCTTTCCGGTTACCATGACCACGACATCCACAGCCTGCTGAACCAGTCGAACCAAATC
CAGAACCAAGTGCTGAACGGTCCCCATGGTTACGGCGTCGACGGTGGGAACGAGCTTGAC
ACTTCACTGTCAAACTCGCTGCTCTCGCTCATGATGGACACTCCGACGTCGCGGTGTACG
GAAGTGCGGACGTTGGACGCCATCGGGCCGAACCCGTTAGCGGAAGTGGACGGAATGTCC
AGCCGGAGTATAATTCATTCCATTTTGACGCGTCGACTCGGACTGATCGTGGGTTGTTGC
TTGGGGATTATCGTTTTTATCGTGCTAATTTCGGTGCTGGGCTATTTGAAGATCAAGAAG
CAGCGGCTAGATGCGGCCAAGCGGCTCCAGCAGCCGCCGATGGCACCGGAATTCATCTCG
TATCGACACTTTTCCATCCCAAACGATGAACACGGTCGGGACGGAGTAGTTGGTGGAGGA
GGGACAAACACGTTCCTGCAGGGGGCCACGGTGGTGGCCAATCATGACGGGCATCCGAGC
TTCATTTCCGGGGCTGTGCTAGGGACGAC

#### The parsed results from the parse_codeml_output.py python script

In [6]:
import pandas as pd

OG0009798_output = pd.read_csv("./TestPAMlOutputs/OG0009798.output", sep='\t')
OG0009798_output

,Gene_1,Gene_2,dnds,dN,dS
0,Culex_pipiens_00001482-RA,Clemson_tarsalis_Ct.00g367850.m01.polypeptide,0.1059,0.1012,0.9563
1,Culex_quinquefasciatus_CPIJ006310-PA,Clemson_tarsalis_Ct.00g367850.m01.polypeptide,0.1201,0.1014,0.8443
2,Culex_quinquefasciatus_CPIJ006310-PA,Culex_pipiens_00001482-RA,0.0010,0.0001,0.0828
3,Davis_tarsalis_mRNA1419,Clemson_tarsalis_Ct.00g367850.m01.polypeptide,0.2674,0.0977,0.3654
4,Davis_tarsalis_mRNA1419,Culex_pipiens_00001482-RA,0.0129,0.0085,0.6611
5,Davis_tarsalis_mRNA1419,Culex_quinquefasciatus_CPIJ006310-PA,0.0140,0.0085,0.6089


[Back To Top](#Top)
***
<a class="anchor" id="4B"></a>
### <span style="color:DodgerBlue">dN/dS Analysis</span>
For this section, the output files were concatinated for analysis.
#### Concatination/File Handling

In [3]:
%%script bash --bg
for infile in *.output
do
     echo ${infile} >> TestFilenames.txt
done

In [ ]:
%%script python3 --bg
with open('TestFilenames.txt') as allfile:
    for line in allfile:
        with open(line[:-1]) as infile:
            with open(line[:-8] + '.new.output','w') as outfile:
                for l in infile:
                    if l.startswith('Gene_1'):
                        pass
                    elif l.startswith('Culex_pipiens'):
                        lr = l.replace('Culex_pipiens',line[:-8] + '_Culex_pipiens')
                        outfile.write(lr)
                    elif l.startswith('Culex_quinquefasciatus'):
                        lr = l.replace('Culex_quinquefasciatus',line[:-8] + '_Culex_quinquefasciatus')
                        outfile.write(lr)
                    elif l.startswith('Davis_tarsalis'):
                        lr = l.replace('Davis_tarsalis',line[:-8] + '_Davis_tarsalis')
                        outfile.write(lr)
                    elif l.startswith('Clemson_tarsalis'):
                        lr = l.replace('Clemson_tarsalis',line[:-8] + '_Clemson_tarsalis')
                        outfile.write(lr)

In [ ]:
%%script bash --bg
cat *.new.output >> AllOG_dNdS.csv

In [8]:
AllOG_output = pd.read_csv("./TestPAMlOutputs/AllOG_dNdS.csv", sep='\t', header=None, names=['Gene_1', 'Gene_2', 'dNdS', 'dN', 'dS'])
AllOG_output

,Gene_1,Gene_2,dNdS,dN,dS
0,OG0007300_Culex_pipiens_00004732-RA,Clemson_tarsalis_Ct.00g115960.m01.polypeptide,0.0010,0.0004,0.3954
1,OG0007300_Culex_quinquefasciatus_CPIJ007475-PA,Clemson_tarsalis_Ct.00g115960.m01.polypeptide,0.0010,0.0004,0.3840
2,OG0007300_Culex_quinquefasciatus_CPIJ007475-PA,Culex_pipiens_00004732-RA,0.0010,0.0001,0.0826
3,OG0007300_Davis_tarsalis_mRNA9767,Culex_pipiens_00004732-RA,0.0079,0.0032,0.3998
4,OG0007300_Davis_tarsalis_mRNA9767,Culex_quinquefasciatus_CPIJ007475-PA,0.0081,0.0032,0.3883
...,...,...,...,...,...
1898,OG0009798_Culex_quinquefasciatus_CPIJ006310-PA,Clemson_tarsalis_Ct.00g367850.m01.polypeptide,0.1201,0.1014,0.8443
1899,OG0009798_Culex_quinquefasciatus_CPIJ006310-PA,Culex_pipiens_00001482-RA,0.0010,0.0001,0.0828
1900,OG0009798_Davis_tarsalis_mRNA1419,Clemson_tarsalis_Ct.00g367850.m01.polypeptide,0.2674,0.0977,0.3654
1901,OG0009798_Davis_tarsalis_mRNA1419,Culex_pipiens_00001482-RA,0.0129,0.0085,0.6611


In [13]:
AllOG_output.sort_values(by="dNdS",ascending=False,inplace=True)
AllOG_output.head(10)

,Gene_1,Gene_2,dNdS,dN,dS
1088,OG0007730_Davis_tarsalis_mRNA6922,Clemson_tarsalis_Ct.00g145430.m01.polypeptide,1.6660,0.0678,0.0407
427,OG0007396_Davis_tarsalis_mRNA729,Clemson_tarsalis_Ct.00g123370.m01.polypeptide,1.3342,0.0454,0.0340
1833,OG0009783_Davis_tarsalis_mRNA14003,Clemson_tarsalis_Ct.00g365570.m01.polypeptide,1.3051,0.0858,0.0658
1471,OG0009706_Davis_tarsalis_mRNA12595,Clemson_tarsalis_Ct.00g345110.m01.polypeptide,1.1717,0.5200,0.4437
283,OG0007361_Culex_quinquefasciatus_CPIJ005712-PA,Culex_pipiens_00007611-RA,1.1373,0.1443,0.1269
585,OG0007529_Davis_tarsalis_mRNA5954,Clemson_tarsalis_Ct.00g132400.m01.polypeptide,0.9780,0.2153,0.2201
198,OG0007346_Davis_tarsalis_mRNA5529,Clemson_tarsalis_Ct.00g120140.m01.polypeptide,0.8922,0.1130,0.1266
1736,OG0009760_Davis_tarsalis_mRNA1251,Clemson_tarsalis_Ct.00g356860.m01.polypeptide,0.8595,0.4014,0.4670
1734,OG0009760_Culex_pipiens_00004482-RA,Clemson_tarsalis_Ct.00g356860.m01.polypeptide,0.8421,0.4000,0.4750
1735,OG0009760_Culex_quinquefasciatus_CPIJ008263-PA,Clemson_tarsalis_Ct.00g356860.m01.polypeptide,0.8421,0.4000,0.4750


In [14]:
AllOG_output.tail(10)

,Gene_1,Gene_2,dNdS,dN,dS
1778,OG0009771_Culex_quinquefasciatus_CPIJ006993-PA,Culex_pipiens_00015116-RA,0.001,0.0002,0.1970
1279,OG0007769_Culex_quinquefasciatus_CPIJ018071-PA,Culex_pipiens_00015863-RA,0.001,0.0001,0.0515
650,OG0007542_Culex_quinquefasciatus_CPIJ005304-PA,Culex_pipiens_00004838-RA,0.001,0.0001,0.0700
1663,OG0009747_Davis_tarsalis_mRNA2270,Culex_pipiens_00014092-RA,0.001,0.0003,0.2971
776,OG0007570_Culex_quinquefasciatus_CPIJ003258-PA,Culex_pipiens_00017449-RA,0.001,0.0001,0.0606
771,OG0007569_Culex_quinquefasciatus_CPIJ003266-PA,Culex_pipiens_00017796-RA,0.001,0.0001,0.0654
1784,OG0009772_Culex_quinquefasciatus_CPIJ013348-PA,Culex_pipiens_00003920-RA,0.001,0.0000,0.0168
735,OG0007562_Culex_quinquefasciatus_CPIJ003225-PA,Culex_pipiens_00002797-RA,0.001,0.0001,0.0864
1300,OG0007775_Culex_quinquefasciatus_CPIJ005778-PA,Culex_pipiens_00008680-RA,0.001,0.0000,0.0381
0,OG0007300_Culex_pipiens_00004732-RA,Clemson_tarsalis_Ct.00g115960.m01.polypeptide,0.001,0.0004,0.3954


[Back To Top](#Top)
***